# House Rent Pricer Project

This project explores various approaches to predicting house rent prices in Lagos, Nigeria, using the dataset [`adiscodex/nigeria-lagos-rent-2022`](https://huggingface.co/datasets/adiscodex/nigeria-lagos-rent-2022). The goal is to evaluate both traditional machine learning models and state-of-the-art frontier language models for the rent prediction task.

In [ ]:

import random
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from pricer.evaluator import evaluate
from openai import OpenAI
import os
from pricer.items  import Item
from dotenv import load_dotenv
import plotly.graph_objects as go
load_dotenv(override=True)

In [ ]:
train, val, test = Item.from_hub("adiscodex/nigeria-lagos-rent-2022")
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
training_prices = [item.price for item in train]
training_average = sum(training_prices) / len(training_prices)
print(training_average)

def constant_pricer(item):
    return training_average

evaluate(constant_pricer, test)

In [ ]:
np.random.seed(42)

prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X = vectorizer.fit_transform(documents)

selected_words = vectorizer.get_feature_names_out()
print(f"Number of selected words: {len(selected_words)}")
print("Selected words:", selected_words[1000:1020])

In [ ]:
regressor = LinearRegression()
regressor.fit(X, prices)

def natural_language_linear_regression_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(regressor.predict(x)[0], 0)

evaluate(natural_language_linear_regression_pricer, test)

In [ ]:
subset = 15_000
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X[:subset], prices[:subset])

def random_forest(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

evaluate(random_forest, test)

In [ ]:
import xgboost as xgb

np.random.seed(42)

xgb_model = xgb.XGBRegressor(n_estimators=5000, random_state=42, n_jobs=4, learning_rate=0.1)
xgb_model.fit(X, prices)

def xg_boost(item):
    x = vectorizer.transform([item.summary])
    return max(0, xgb_model.predict(x)[0])

evaluate(xg_boost, test)

# And now - to the frontier!

Let's see how Frontier models do out of the box; no training, just inference based on their world knowledge.

In [ ]:
OPEN_ROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')

client = OpenAI(
    base_url=OPEN_ROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY
)

def messages_for(item):
    message = Item.estimate_price_prompt(item)
    return [{"role": "user", "content": message}]

print(test[0].summary)
print(messages_for(test[0]))

In [ ]:
def gpt_4__1_nano(item):
    response = client.chat.completions.create(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

evaluate(gpt_4__1_nano, test)

In [ ]:
def claude_opus_4_5(item):
    response = client.chat.completions.create(model="anthropic/claude-opus-4.5", messages=messages_for(item))
    return response.choices[0].message.content

evaluate(claude_opus_4_5, test)

In [ ]:
def gemini_2_5_flash(item):
    response = client.chat.completions.create(model="google/gemini-2.5-flash", messages=messages_for(item))
    return response.choices[0].message.content

evaluate(gemini_2_5_flash, test)

## Testing the Fine-Tuned GPT-4.1 Nano Model

Let's evaluate the performance of the fine-tuned GPT-4.1 Nano model on the house rent pricing task. The following cell will use the `gpt_4__1_nano_fine_tuned` function to predict prices for the test set and display the evaluation metrics.

In [ ]:
openai = OpenAI()

fine_tuned_model_name = "ft:gpt-4.1-nano-2025-04-14:personal:pricer:DGNmQ7TR"

def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content

evaluate(gpt_4__1_nano_fine_tuned, test)

In [ ]:
results = [
    ("Constant", "gray", 1_105_493),
    ("NLP + LR", "gray", 948_924),
    ("Random Forest", "gray", 966_450),
    ("XGBoost", "gray", 876_821),
    ("GPT 4.1 Nano", "slateblue", 2_447_859),
    ("Claude 4.5 Opus", "slateblue", 1_249_100),
    ("Gemini 2.5 Flash", "slateblue", 1_244_850),
    ("GPT 4.1 Nano (Fine-tuned)", "skyblue", 2_134_425),
]

labels, colors, values = zip(*results)

fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors))

fig.update_layout(
    title="Prediction error from each model",
    yaxis=dict(range=[0, max(values)], title="Error"),
    xaxis=dict(tickangle=-45),
    width=1000,
    height=800
)

fig.show()